# Notebook 2 — Sector-Conditioned Head (Path C)**Goal:** train a lightweight 2-stage head on top of saved ModernBERT-large CLS embeddings. Stage 1 predicts the sector (~11 classes). Stage 2 predicts the industry conditional on the sector. The hierarchical structure attacks the long-tail macro F1 problem directly.**Expected outcome:** +3–5 macro F1 points over the single best fine-tune (70.29% → ~73–75%) and a stronger top-3.**Hardware:** CPU works; T4 makes it faster. No fine-tuning of the encoder, just a small MLP head.Sequenced steps:1. Mount + load saved CLS embeddings from the best run2. Build the code → sector mapping from gecs_taxonomy.json3. Carve a stratified dev split out of the training embeddings (the head must NOT see test until the final eval)4. Define the hierarchical head model5. Train with combined sector + industry-conditional loss; tune on dev6. Evaluate ONCE on the test embeddings; report macro F1 + top-k7. Save predictions + checkpoint

In [ ]:
# === Mount + imports ===
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
# === CONFIG ===
CONFIG = {
    # Path to the run whose CLS embeddings we'll use as the feature source.
    # Use the BEST single-model run from Notebook 1's per-run table.
    'BEST_RUN_DIR':  '/content/drive/MyDrive/v3_segaware_joint_s42',  # ← UPDATE if a different run was best
    'TAXONOMY_PATH': '/content/drive/MyDrive/gecs_taxonomy.json',
    # OR upload taxonomy locally and use '/content/gecs_taxonomy.json'
    'OUTPUT_DIR':    '/content/drive/MyDrive/v3_sector_head',
    # Training
    'DEV_FRAC':       0.10,
    'BATCH_SIZE':     512,
    'EPOCHS':         25,
    'LR':             3e-4,
    'HIDDEN_DIM':     768,
    'DROPOUT':        0.2,
    # Loss weighting: industry head is primary
    'SECTOR_W':       0.3,
    'INDUSTRY_W':     1.0,
    'SEED':           42,
}
os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)
torch.manual_seed(CONFIG['SEED'])
np.random.seed(CONFIG['SEED'])
print('Config locked in.')


In [ ]:
# === Step 1: Load saved embeddings and metadata ===
train_cls = np.load(os.path.join(CONFIG['BEST_RUN_DIR'], 'train_cls.npy'))
test_cls  = np.load(os.path.join(CONFIG['BEST_RUN_DIR'], 'test_cls.npy'))
classes   = np.load(os.path.join(CONFIG['BEST_RUN_DIR'], 'industry_classes.npy'), allow_pickle=True)
train_meta = pd.read_csv(os.path.join(CONFIG['BEST_RUN_DIR'], 'train_meta.csv'))
test_meta  = pd.read_csv(os.path.join(CONFIG['BEST_RUN_DIR'], 'test_meta.csv'))

print(f'Train embeddings: {train_cls.shape}')
print(f'Test embeddings:  {test_cls.shape}')
print(f'Classes: {len(classes)} (first 5: {list(classes[:5])})')
print(f'Train meta cols:  {list(train_meta.columns)}')
print(f'Test meta cols:   {list(test_meta.columns)}')


In [ ]:
# === Step 2: Build code → sector mapping ===
# GECS hierarchy: 8-digit Morningstar code → sector (first 2 digits) → industry group (4) → industry (6 or 8).
# The exact roll-up depends on the taxonomy file. We try the file first, then fall back to a prefix rule.

code_to_sector = {}
try:
    with open(CONFIG['TAXONOMY_PATH']) as f:
        tax = json.load(f)
    # Expect structure like {sector_id: {name: ..., industries: {ind_id: {...}}}} or similar
    # Walk it and build the mapping
    def walk(node, parent_sector=None):
        if isinstance(node, dict):
            for k, v in node.items():
                if isinstance(v, dict):
                    # Heuristic: 2-digit keys at the top are sectors
                    sector_here = k if (parent_sector is None and len(str(k)) == 2) else parent_sector
                    walk(v, sector_here)
                elif isinstance(v, (list, tuple)):
                    for item in v:
                        walk(item, parent_sector)
                else:
                    # Leaf: if key looks like a Morningstar code and we have a sector, map it
                    if parent_sector is not None and len(str(k)) >= 6:
                        code_to_sector[str(k)] = parent_sector
        elif isinstance(node, list):
            for item in node:
                walk(item, parent_sector)
    walk(tax)
    print(f'From taxonomy file: {len(code_to_sector)} code→sector mappings')
except FileNotFoundError:
    print(f'Taxonomy file not found at {CONFIG["TAXONOMY_PATH"]}. Falling back to prefix rule.')

# Fallback: use the first 2 digits of the Morningstar code as the sector.
# This works because Morningstar GECS codes are hierarchical by digit position.
if not code_to_sector:
    for c in classes:
        code_to_sector[str(c)] = str(c)[:2]
    print(f'Built {len(code_to_sector)} mappings using 2-digit-prefix rule.')

# Build sector list and indexing
sectors = sorted(set(code_to_sector.values()))
sector_to_idx = {s: i for i, s in enumerate(sectors)}
print(f'\nFound {len(sectors)} sectors: {sectors}')

# Build industry_idx -> sector_idx lookup (used by the head at inference)
industry_to_sector = np.zeros(len(classes), dtype=np.int64)
for i, c in enumerate(classes):
    sec = code_to_sector.get(str(c), sectors[0])
    industry_to_sector[i] = sector_to_idx[sec]
print(f'industry_to_sector array shape: {industry_to_sector.shape}')


In [ ]:
# === Step 3: Build labels and dev split ===
# Train labels (industry idx)
LABEL_COL_CANDIDATES = ['label_idx', 'industry_label', 'mstar_code', 'label']
def pick_label_col(df):
    for c in LABEL_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise RuntimeError(f'No label column in: {list(df.columns)}')

train_label_col = pick_label_col(train_meta)
test_label_col  = pick_label_col(test_meta)
print(f'Train label col: {train_label_col} | Test label col: {test_label_col}')

classes_to_idx = {str(c): i for i, c in enumerate(classes)}
def to_int_labels(df, col):
    if col == 'label_idx':
        return df[col].astype(int).values
    return df[col].astype(str).map(classes_to_idx).fillna(-1).astype(int).values

y_train_ind = to_int_labels(train_meta, train_label_col)
y_test_ind  = to_int_labels(test_meta,  test_label_col)
# Drop any unmapped rows (just in case)
mask_tr = y_train_ind >= 0
mask_te = y_test_ind >= 0
train_cls = train_cls[mask_tr]; y_train_ind = y_train_ind[mask_tr]
test_cls  = test_cls[mask_te];  y_test_ind  = y_test_ind[mask_te]
print(f'After mapping: train={len(y_train_ind)}, test={len(y_test_ind)}')

y_train_sec = industry_to_sector[y_train_ind]
y_test_sec  = industry_to_sector[y_test_ind]

# Stratified dev split (do not touch test embeddings yet)
X_tr, X_dev, yi_tr, yi_dev, ys_tr, ys_dev = train_test_split(
    train_cls, y_train_ind, y_train_sec,
    test_size=CONFIG['DEV_FRAC'],
    random_state=CONFIG['SEED'],
    stratify=y_train_sec  # stratify by sector — robust even for tiny industries
)
print(f'Head training set:  {X_tr.shape}')
print(f'Head dev set:       {X_dev.shape}')


In [ ]:
# === Step 4: Hierarchical head model ===
class HierarchicalHead(nn.Module):
    def __init__(self, in_dim, n_sectors, n_industries, hidden, dropout):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.sector_head = nn.Linear(hidden, n_sectors)
        self.industry_head = nn.Linear(hidden, n_industries)
        # Buffer: industry → sector mapping for conditional masking
        ind_to_sec = torch.from_numpy(industry_to_sector).long()
        self.register_buffer('ind_to_sec', ind_to_sec)
        self.n_sectors = n_sectors

    def forward(self, x):
        h = self.shared(x)
        sec_logits = self.sector_head(h)
        ind_logits = self.industry_head(h)
        return sec_logits, ind_logits

    def conditional_predict(self, x):
        """Predict industry conditional on the predicted sector — gate industry logits."""
        sec_logits, ind_logits = self.forward(x)
        sec_probs = F.softmax(sec_logits, dim=-1)            # (B, n_sectors)
        # For each industry, its sector probability → multiplier
        ind_sec_probs = sec_probs[:, self.ind_to_sec]        # (B, n_industries)
        ind_probs = F.softmax(ind_logits, dim=-1) * ind_sec_probs
        return ind_probs / (ind_probs.sum(dim=-1, keepdim=True) + 1e-12)

model = HierarchicalHead(
    in_dim=train_cls.shape[1],
    n_sectors=len(sectors),
    n_industries=len(classes),
    hidden=CONFIG['HIDDEN_DIM'],
    dropout=CONFIG['DROPOUT'],
).to(device)
print(model)


In [ ]:
# === Step 5: Training loop ===
def make_loader(X, yi, ys, shuffle):
    ds = TensorDataset(
        torch.from_numpy(X).float(),
        torch.from_numpy(yi).long(),
        torch.from_numpy(ys).long(),
    )
    return DataLoader(ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=shuffle, num_workers=0)

tr_loader  = make_loader(X_tr, yi_tr, ys_tr, shuffle=True)
dev_loader = make_loader(X_dev, yi_dev, ys_dev, shuffle=False)

opt = torch.optim.AdamW(model.parameters(), lr=CONFIG['LR'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG['EPOCHS'])
ce = nn.CrossEntropyLoss()

@torch.no_grad()
def eval_loader(loader):
    model.eval()
    preds, trues = [], []
    for x, yi, ys in loader:
        x = x.to(device)
        p = model.conditional_predict(x).cpu().numpy()
        preds.append(p.argmax(axis=1))
        trues.append(yi.numpy())
    return np.concatenate(preds), np.concatenate(trues)

best_dev_f1 = 0
best_state = None
for epoch in range(CONFIG['EPOCHS']):
    model.train()
    total = 0
    for x, yi, ys in tr_loader:
        x, yi, ys = x.to(device), yi.to(device), ys.to(device)
        sec_logits, ind_logits = model(x)
        loss = CONFIG['SECTOR_W'] * ce(sec_logits, ys) + CONFIG['INDUSTRY_W'] * ce(ind_logits, yi)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    scheduler.step()
    # Dev eval (conditional prediction)
    dev_preds, dev_trues = eval_loader(dev_loader)
    dev_f1 = f1_score(dev_trues, dev_preds, average='macro', zero_division=0)
    flag = ''
    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        flag = ' ★'
    print(f'Epoch {epoch+1:2d}/{CONFIG["EPOCHS"]} | loss {total/len(tr_loader):.4f} | dev macro F1 {dev_f1*100:.2f}%{flag}')

print(f'\nBest dev macro F1: {best_dev_f1*100:.2f}%')
model.load_state_dict(best_state)


In [ ]:
# === Step 6: Final test evaluation (only now we touch test embeddings) ===
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(test_cls).float(), torch.from_numpy(y_test_ind).long()),
    batch_size=CONFIG['BATCH_SIZE'], shuffle=False
)
model.eval()
all_probs = []
with torch.no_grad():
    for x, _ in test_loader:
        x = x.to(device)
        all_probs.append(model.conditional_predict(x).cpu().numpy())
test_probs = np.concatenate(all_probs, axis=0)
test_top1 = test_probs.argmax(axis=1)

def topk_acc(probs, y, k):
    topk = np.argsort(-probs, axis=1)[:, :k]
    return float(np.any(topk == y[:, None], axis=1).mean())

print('=== TEST SET — sector-conditioned head ===')
print(f'  Macro F1:    {f1_score(y_test_ind, test_top1, average="macro", zero_division=0)*100:.2f}%')
for k in [1, 3, 5]:
    print(f'  Top-{k} acc:  {topk_acc(test_probs, y_test_ind, k)*100:.2f}%')


In [ ]:
# === Step 7: Save predictions + checkpoint ===
np.save(os.path.join(CONFIG['OUTPUT_DIR'], 'sector_head_probs.npy'), test_probs)
np.save(os.path.join(CONFIG['OUTPUT_DIR'], 'sector_head_top1.npy'), test_top1)
torch.save(best_state, os.path.join(CONFIG['OUTPUT_DIR'], 'sector_head.pt'))
with open(os.path.join(CONFIG['OUTPUT_DIR'], 'summary.json'), 'w') as f:
    json.dump({
        'best_dev_macro_f1': float(best_dev_f1),
        'test_macro_f1': float(f1_score(y_test_ind, test_top1, average='macro', zero_division=0)),
        'test_top1_acc':  float(topk_acc(test_probs, y_test_ind, 1)),
        'test_top3_acc':  float(topk_acc(test_probs, y_test_ind, 3)),
        'test_top5_acc':  float(topk_acc(test_probs, y_test_ind, 5)),
        'source_run': CONFIG['BEST_RUN_DIR'],
        'n_sectors': len(sectors),
        'n_industries': len(classes),
    }, f, indent=2)
print('Saved to', CONFIG['OUTPUT_DIR'])


In [ ]:
# === Step 8 (bonus): Ensemble sector head with Notebook 1 ensemble ===
# If the Notebook 1 ensemble probs are on disk, combine 50/50 here for the strongest single number.
ENSEMBLE_PROBS = '/content/drive/MyDrive/v3_ensemble_results/ensemble_probs.npy'
if os.path.exists(ENSEMBLE_PROBS):
    e_probs = np.load(ENSEMBLE_PROBS)
    n = min(len(e_probs), len(test_probs))
    combined = 0.5 * e_probs[:n] + 0.5 * test_probs[:n]
    combined_top1 = combined.argmax(axis=1)
    print('=== ENSEMBLE + SECTOR HEAD (50/50) ===')
    print(f'  Macro F1:    {f1_score(y_test_ind[:n], combined_top1, average="macro", zero_division=0)*100:.2f}%')
    for k in [1, 3, 5]:
        print(f'  Top-{k} acc:  {topk_acc(combined, y_test_ind[:n], k)*100:.2f}%')
    np.save(os.path.join(CONFIG['OUTPUT_DIR'], 'ensemble_plus_sector_probs.npy'), combined)
else:
    print('No Notebook 1 ensemble probs found — skipping combined eval.')


## Next steps after Notebook 2If the combined ensemble + sector-head macro F1 is:- **≥ 75%** → goal reached. Move to writing it up.- **73–75%** → run Notebook 3 (class-balanced loss fine-tune) for the final push.- **< 73%** → revisit the sector-mapping (the taxonomy walk may have misidentified sectors). Confirm `industry_to_sector` array is sensible by spot-checking a few codes.